# Forecasting model improvement - saved evidence

**Run All only displays saved reports. It never trains, fits weights, calibrates or reopens holdout.**

Main objective: mean normalized original-unit MAE for 1h/6h/24h. The inner folds stop and score on separate chronological windows. The outer audit is previously seen validation 504-671, not a fresh test. Production and final holdout remain unchanged.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

roots = list(Path('/kaggle/input').rglob('improvement_audit.csv')) if Path('/kaggle/input').exists() else []
REPORTS = roots[0].parent if roots else Path('reports')
if not (REPORTS/'improvement_audit.csv').exists():
    raise FileNotFoundError('Attach the completed improvement audit outputs')


## 1. Development folds

The scoring period is never used for early stopping within its fold. Some LightGBM candidates are screened out after inner1.


In [ ]:
lgbm = pd.read_csv(REPORTS/'improvement_lgbm_cv.csv')
catboost = pd.read_csv(REPORTS/'improvement_catboost_cv.csv')
cv = pd.concat([lgbm,catboost],ignore_index=True)
display(cv.pivot_table(index=['fold','model'],columns='horizon',values='mae'))


## 2. Inner-only ensemble choice

Weights and a small bias grid were calibrated on cloud using only inner scores through hour 503. They were saved before audit construction.


In [ ]:
selection = json.loads((REPORTS/'improvement_selection.json').read_text())
display(pd.DataFrame([{'horizon':h,'inner_mae':r['mae'],'bias':r['bias'],'members':' + '.join(f'{n}: {w:.1f}' for n,w in r['weights'].items())} for h,r in selection.items()]))


## 3. Reused-validation audit

Lower MAE is better. Compare only matching horizons and common finite pairs. This is not a new final holdout result.


In [ ]:
audit = pd.read_csv(REPORTS/'improvement_audit.csv')
pooled = audit[audit.load_tier == 'All']
display(pooled[['model','horizon','mae','rmse','wape','n_valid']])
mae = pooled.pivot(index='model',columns='horizon',values='mae')
display((100*(1-mae/mae.loc['lightgbm_c1'])).round(3).rename_axis('MAE reduction vs C1 (%)'))


## 4. Paired BS uncertainty

Difference is improved ensemble MAE minus reference MAE. Negative favors the ensemble on this sample; it does not establish unseen-week generalization.


In [ ]:
paired = pd.read_csv(REPORTS/'improvement_paired.csv')
display(paired[['horizon','reference','difference','lower','upper','status']])


## 5. Risk and zero/active behavior

P90/P95 policy is unchanged and fitted only through hour 503. Precision, recall and FPR must be read together. Strict zero-FP (`point > 0` when actual is 0) is a separate metric.


In [ ]:
risk = pd.read_csv(REPORTS/'improvement_risk.csv')
display(risk[['model','horizon','threshold','precision','recall','fpr','positive_events']])
display(pooled[['model','horizon','active_only_mae','zero_fp_rate']])


## 6. Load-group errors

Load groups use fit data only. Inspect low-load WAPE and high-load absolute error alongside pooled MAE.


In [ ]:
display(audit.loc[audit.load_tier != 'All', ['model','horizon','load_tier','mae','wape']])


## Scope and next evidence

These results support a research candidate, not a replacement final benchmark. The production artifact directory is untouched. Fresh unseen chronological evaluation data is required for a new frozen promotion. All training, including tiny fitting tests and blend calibration, ran on hosted Kaggle. Checkpoints remain in Kaggle Output; source code is in the private project repository.
